# 第03课 - 代理设计模式

在本课中，我们将探索构建高效 AI 代理的三个基础设计模式：

1. <strong>清晰的代理指令</strong> — 制作精确的、定义角色的提示，以指导代理行为
2. **使用 Pydantic 模型的结构化输出** — 确保代理返回可预测、已验证的数据
3. <strong>单一职责代理</strong> — 设计专注的代理，每个代理专注做好一件事

我们将把每种模式应用于一个<strong>旅游目的地推荐系统</strong>场景，逐步构建一个能够推荐目的地、检查可用性和处理物流的系统。


## 设置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 模式1：明确的代理指令

最有影响力的模式也是最简单的：为你的代理编写清晰、详细的指令。

良好的指令应定义：
- <strong>代理是谁</strong>（角色和语气）
- <strong>代理该做什么</strong>（逐步职责）
- <strong>代理应如何表现</strong>（约束和风格）

下面，我们创建一个旅行礼宾代理，带有明确的指令来塑造它生成的每个回复。


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## 模式 2：使用 Pydantic 模型的结构化输出

自由形式文本对对话很有用，但下游系统需要结构化数据。
通过将 **Pydantic 模型** 与 <strong>工具函数</strong> 配对，我们可以：

- 定义代理输出的精确定义模式
- 自动验证响应
- 可靠地将代理结果集成到应用逻辑中

执行的关键是在运行代理时传递 `response_format`。这会强制
模型返回一个经过验证的 `TravelRecommendations` 对象（可通过 `response.value` 访问）
，而不是自由格式文本。`get_destination_details` 工具也返回类型化的
`DestinationRecommendation`，因此数据从始至终保持结构化。


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## 模式 3：单一职责代理

复杂任务通过将工作拆分为多个专注的代理来执行，每个代理负责单一职责：

- 一个了解地点和可用性的 <strong>目的地专家</strong>
- 一个处理航班、酒店和行程的 <strong>物流规划师</strong>

这与软件工程中的<em>关注点分离</em>原则相呼应——每个代理都更容易独立测试、维护和改进。


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## 总结

在本课中，我们将三个主动设计模式应用于旅游推荐场景：

| 模式 | 关键思想 | 优势 |
|---|---|---|
| <strong>明确指令</strong> | 预先定义角色、职责和约束 | 保持一致、符合品牌形象的代理行为 |
| <strong>结构化输出</strong> | 使用Pydantic模型作为响应格式 | 经过验证、机器可读的结果 |
| <strong>单一职责</strong> | 让每个代理专注于一项工作 | 更易测试、维护和组合 |

这些模式自然组合——你可以将明确指令与结构化输出结合到单一职责代理中，构建健壮、适合生产的系统。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
